In [10]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from typing import Any, Dict, Literal
from sklearn.metrics import r2_score
import math
import torch
import torchgeometry as tgm
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib.data_utils import seed_worker, pad_collate
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 
from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)
from inference_full_tile import Dataset_from_files
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#### Utils functions

In [11]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


def _to_cpu(x):
    """Copie récursive de tenseurs en CPU (détachés du graph)."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch


def plot_seq(batch, pred, c_index, BRIGHTNESS_FACTOR=3, t_sampled=None):
    if t_sampled is None:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
            ],
            dim=0,
        )
    else:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
            ],
            dim=0,
        )
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    ncols = int(images.shape[0] / 3)
    fig, axes = plt.subplots(nrows=3, ncols=ncols, figsize=(15, 7))
    column_labels = [f't{i + 1}' for i in range(ncols + 1)]
    for idx, ax in enumerate(axes.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1, 
                                   transform=ax.transAxes,
                                   fill=False, 
                                   edgecolor='black', 
                                   linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    plt.subplots_adjust(wspace=0.05, hspace=0.1)
    
    row_labels = ['Target', 'Inputs', 'Predictions']
    for i, label in enumerate(row_labels):
        axes[i, 0].annotate(label, 
                            xy=(-0.1, 0.5),
                            xycoords='axes fraction',
                            fontsize=10, 
                            fontweight='bold',
                            ha='right',
                            va='center',
                            rotation=90)
    
    plt.subplots_adjust(left=0.18)
    plt.show()


def plot_seq_RGB_NIR(batch, pred, n_visible=8):
    # plot R-G-B
    plot_seq_interactive(batch, pred, c_index=[2, 1, 0], BRIGHTNESS_FACTOR=3, n_visible=n_visible)
    # plot NIR-R-G
    plot_seq_interactive(batch, pred, c_index=[6, 2, 1], BRIGHTNESS_FACTOR=2, n_visible=n_visible)

# ── Versions interactives avec slider ──────────────────────────────────────────

def _draw_grid(images, ncols, t_offset, row_labels, ax_array, fig, BRIGHTNESS_FACTOR):
    """Dessine la grille Target / Inputs / Predictions pour un sous-ensemble de timesteps."""
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    column_labels = [f't{t_offset + i}' for i in range(ncols)]
    for idx, ax in enumerate(ax_array.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1,
                                 transform=ax.transAxes,
                                 fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    for i, label in enumerate(row_labels):
        ax_array[i, 0].annotate(label, xy=(-0.1, 0.5), xycoords='axes fraction',
                                fontsize=10, fontweight='bold',
                                ha='right', va='center', rotation=90)


def plot_seq_interactive(batch, pred, c_index, BRIGHTNESS_FACTOR=3, n_visible=8):
    """Version interactive de plot_seq avec un slider temporel.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    batch = _to_cpu(batch)
    pred = _to_cpu(pred)

    target = batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs = batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds  = pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)
        images = torch.concatenate([target[idx], inputs[idx], preds[idx]], dim=0)
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=3, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 7))
            _draw_grid(images, n_visible, t0,
                       ['Target', 'Inputs', 'Predictions'],
                       axes, fig, BRIGHTNESS_FACTOR)
            plt.subplots_adjust(left=0.10, wspace=0.05, hspace=0.1)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()


def plot_seq_RGB_NIR_interactive(
    targets, 
    inputs,
    preds,
    n_visible=8):
    """Version interactive de plot_seq_RGB_NIR avec un slider partagé RGB / NIR.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    targets = _to_cpu(targets)
    inputs = _to_cpu(inputs)
    preds = _to_cpu(preds)

    target_rgb = targets[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_rgb = inputs[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_rgb  = preds[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)

    target_nir = targets[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_nir = inputs[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_nir  = preds[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target_rgb.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)

        imgs_rgb = torch.concatenate([target_rgb[idx], inputs_rgb[idx], preds_rgb[idx]], dim=0)
        imgs_nir = torch.concatenate([target_nir[idx], inputs_nir[idx], preds_nir[idx]], dim=0)

        row_labels = ['Target', 'Inputs', 'Predictions']
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=6, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 13))
            _draw_grid(imgs_rgb, n_visible, t0, row_labels,
                       axes[:3], fig, BRIGHTNESS_FACTOR=3)
            _draw_grid(imgs_nir, n_visible, t0, row_labels,
                       axes[3:], fig, BRIGHTNESS_FACTOR=2)

            # Titres de section
            fig.text(0.02, 0.78, 'RGB', fontsize=14, fontweight='bold',
                     rotation=90, va='center')
            fig.text(0.02, 0.35, 'NIR-R-G', fontsize=14, fontweight='bold',
                     rotation=90, va='center')

            plt.subplots_adjust(left=0.08, wspace=0.05, hspace=0.15)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()

def getitem_from_mgrsc(df, mgrsc, window: str = None):
    if window is None:
        return df[df["mgrs25"] == mgrsc].index.values
    else:
        return df[(df["mgrs25"] == mgrsc) & (df["window"] == window)].index.values[0]

## Inférences faites depuis le fichier hdf5 

In [12]:
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)
cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.8
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3
phase = "test"

path_inference_config_file = Path("/home/SPeillet/rpg_3STR/cloud_reconstruction/U-TILISE/configs/config_run_eval_multi_stream.yaml")
# path_ckpt_config_file = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/config.yaml")
# path_ckpt_pth = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/checkpoints/Model_best.pth")
path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/config.yaml")
path_ckpt_pth = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/checkpoints/Model_best.pth")


training_config_file = config_utils.read_config(path_ckpt_config_file)
inference_config_file = config_utils.read_config(path_inference_config_file)

temporal_window = 6
# Faire la différence entre l'imputation d'une TS (avec plusieurs intervalles de dates) et plusieurs observations.
inference_imputation = Imputation(
    config_file_train=path_inference_config_file, # Config permettant de faire la configuration de l'inference
    method="utilise",
    checkpoint=path_ckpt_pth,
    config_file_test=path_ckpt_config_file, # Fichier ayant servi à l'entrainement du modèle (update params model)
    temporal_window=temporal_window,
)

dset = data_utils.get_dataset(config, phase=phase)

# dset = torch.utils.data.Subset(dset, range(SUBSET_LENGTH))
dataloader = torch.utils.data.DataLoader(
    dataset=dset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)



Loaded transforms from ./data/CIRCA_patches_datasets_with_transforms.json.
Checkpoint '/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/checkpoints/Model_best.pth' loaded.
Chosen epoch: 38



In [13]:
item = 500
t_sampled = [2, 4, 6, 8, 9, 10, 13]
t_masked = {"indices_masked": [1, 2, 4, 6]}
# t_sampled = [34, 35, 36, 37, 38, 39, 40, 41, 42]
# t_masked = None

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl1 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample1 = dl1.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample1["info"])
batch1 = sample_to_batch(sample1)

batch1, y_pred, att = inference_imputation.impute_sample(
    batch=batch1,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

# plot_seq_RGB_NIR(batch1, y_pred)
plot_seq_RGB_NIR_interactive(
targets=batch1["y"][0],
inputs=batch1["x"][0],
preds=y_pred[0],
) 

{'mgrs': '30UXV', 'mgrs25': '30UXV_row-2_col-2', 'window': '1536_1940_256_256'}


## Inférence faite depuis les fichiers du store-dai

### Good sample

In [16]:
store_dai = Path("/mnt/stores/store_dai")
path_dataset_circa = store_dai / "projets/pac/3str/EXP_2/Data_Raster"
data_optique = path_dataset_circa / "optique_dataset"
data_radar = path_dataset_circa / "radar_dataset_v4"
path_test_set_mgrs25 = store_dai / "projets/pac/3str/EXP_2/train_val_test/MGRSC_test.json"
test_mgrs25 = json.load(open(path_test_set_mgrs25))
# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.0
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dict_mgrs = {f.stem: f for f in data_optique.iterdir()}
dict_mgrsc = {p.stem: p for f in dict_mgrs.values() for p in f.iterdir()}

mgrsc_good_sample = '30UXV_row-2_col-2'
window_good_sample = (1536, 1940, 256, 256)
image_size = [256, 256]
OVERLAP = 0

ds_from_files = Dataset_from_files(
    mgrsc=mgrsc_good_sample,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=image_size,
    overlap=OVERLAP,
    fill_value=1.0,
    mask_type='original_masks',
    load_dataset="./tiles_windows.csv",
)

dl_from_files = torch.utils.data.DataLoader(
    dataset=ds_from_files,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

In [17]:
index_good_sample = getitem_from_mgrsc(ds_from_files.mgrsc_dataset, mgrsc=mgrsc_good_sample, window=window_good_sample)
item = index_good_sample
# t_sampled = None
t_sampled = None
t_masked = None

sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 21, 22, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)
# batch_from_files["position_days"] = batch_hdf5["position_days"]
# batch_from_files["masks_valid_obs"] = batch_hdf5["masks_valid_obs"]
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

Original masks shape: torch.Size([7, 2, 256, 256])


In [ ]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=t_sampled)
batch_from_files = sample_to_batch(sample_from_files)

batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [8]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [9]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)

Original masks shape: torch.Size([5, 2, 256, 256])


In [10]:
import numpy as np
import matplotlib.pyplot as plt

def analyze_and_plot_distributions(b_hdf5, b_tif):
    """
    Calcule et compare les distributions de pixels par grandes catégories
    (Optique S2 et potentiellement Radar S1 si présent) entre les deux approches.
    Rejette les pixels masqués pour ne juger que de la vraie donnée.
    """
    
    # 1. Extraction et nettoyage de 'y' (cibles complètes)
    # y est au format (Batch, Temps, Canaux, H, W). Pour la distribution globale on s'en fiche du temps et x,y.
    # On va regrouper tous les pixels valides.
    
    y_h5  = b_hdf5["y"][0].detach().cpu().numpy()
    y_tif = b_tif["y"][0].detach().cpu().numpy()

    c_s2 = 10 # Nombre de canaux Sentinel-2
    # Séparation Optique S2 (Les 10 premiers canaux)
    s2_h5  = y_h5[:, :c_s2, :, :].flatten()
    s2_tif = y_tif[:, :c_s2, :, :].flatten()
    
    print("="*60)
    print("STATISTIQUES GLOBALES - IMAGES CIBLES (y)")
    print("="*60)
    print(f"{'Source':<15} | {'Min':<8} | {'Max':<8} | {'Moyenne':<8} | {'Médiane':<8} | {'Ecart-type':<8}")
    print("-" * 60)
    print(f"{'HDF5 (S2)':<15} | {s2_h5.min():.4f}   | {s2_h5.max():.4f}   | {s2_h5.mean():.4f}   | {np.median(s2_h5):.4f}   | {s2_h5.std():.4f}")
    print(f"{'TIF_file (S2)':<15} | {s2_tif.min():.4f}   | {s2_tif.max():.4f}   | {s2_tif.mean():.4f}   | {np.median(s2_tif):.4f}   | {s2_tif.std():.4f}")
    
    # Si le radar est présent (Nb Canaux > 10)
    has_sar = y_h5.shape[1] > c_s2
    if has_sar:
        s1_h5 = y_h5[:, c_s2:, :, :].flatten()
        s1_tif = y_tif[:, c_s2:, :, :].flatten()
        print(f"{'HDF5 (S1_SAR)':<15} | {s1_h5.min():.4f}   | {s1_h5.max():.4f}   | {s1_h5.mean():.4f}   | {np.median(s1_h5):.4f}   | {s1_h5.std():.4f}")
        print(f"{'TIF_file(S1_SAR)':<15} | {s1_tif.min():.4f}   | {s1_tif.max():.4f}   | {s1_tif.mean():.4f}   | {np.median(s1_tif):.4f}   | {s1_tif.std():.4f}")
        
    print("\n")
        
    # 2. Visualisation des distributions de densité (Histogrammes)
    fig, axes = plt.subplots(1, 2 if has_sar else 1, figsize=(12 if has_sar else 6, 5))
    
    if not isinstance(axes, np.ndarray):
        axes = [axes]
        
    # Hist Optique
    # On limite à 1.0 au cas où, pour éviter que des valeurs extrêmes (outliers) écrasent le visuel
    bins_S2 = np.linspace(0, min(1.0, max(s2_h5.max(), s2_tif.max())), 100) 
    
    axes[0].hist(s2_h5, bins=bins_S2, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
    axes[0].hist(s2_tif, bins=bins_S2, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
    axes[0].set_title('Densité des valeurs des pixels Sentinel-2')
    axes[0].set_xlabel('Valeur normalisée (0 à 1)')
    axes[0].set_ylabel('Densité')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Hist Radar si applicable
    if has_sar:
        bins_S1 = np.linspace(0, 1.0, 100)
        axes[1].hist(s1_h5, bins=bins_S1, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
        axes[1].hist(s1_tif, bins=bins_S1, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
        axes[1].set_title('Densité des valeurs des pixels Sentinel-1 (SAR)')
        axes[1].set_xlabel('Valeur normalisée (0 à 1)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Exécution du test de compatibilité distributionnelle
analyze_and_plot_distributions(batch_hdf5, batch_from_files)

NameError: name 'batch_hdf5' is not defined

In [ ]:
batch_hdf5.keys()

In [ ]:
batch_hdf5["position_days"]

In [ ]:
batch_hdf5["position_days"]

### Observation direct des inférences en mode produit

In [ ]:
from rasterio.windows import Window
from dataloader_CIRCA.tools.mask_generation import masks_init_filling
import torch

def read_data_by_window(
    target_window,
    path_file,
):
    """
        Return a ndarray of a crop tile with shape T * H * W * C
    """
    with rasterio.open(path_file) as src:
        array = src.read(window=Window(*target_window))
        array = array.reshape((int(array.shape[0] // 12), 12, array.shape[1], array.shape[2])).astype(np.float32)
        return array[:, :10, ...], array[:, -1, ...]

In [ ]:
mgrsc_target = '30UXV_row-2_col-2'
target_window = (1536, 1940, 256, 256)
# mgrsc_target = mgrsc_bad_sample = "30TYS_row-4_col-4"
# target_window = window_bad_sample = (1536, 512, 256, 256)

path_preds = store_dai / "tmp/speillet/inferences"
pred_file = path_preds / f"pred_mgrsc_{mgrsc_target}.tif"
assert pred_file.exists(), f"File {pred_file} doesn't exists.."
preds, _ = read_data_by_window(
    target_window,
    path_file=pred_file,
)

path_inputs = store_dai / "projets/pac/3str/EXP_2/Data_Raster/optique_dataset"
input_file = path_inputs / mgrsc_target[:5] / f"MGRS25-{mgrsc_target}" /  f"bands_stacked_{mgrsc_target}.tif"
assert input_file.exists(), f"File {input_file} doesn't exists.."
inputs, cloud_mask = read_data_by_window(
    target_window,
    path_file=input_file,
)

inputs, preds, cloud_mask = torch.from_numpy(inputs), torch.from_numpy(preds), torch.from_numpy(cloud_mask)
inputs_masked, masks = masks_init_filling(
    seq=inputs.clone(),
    masks=cloud_mask.clone(),
    fill_type="fill_value",
    fill_value=1,
    dilate_cloud_masks=False,
)

In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=inputs/10000, 
    inputs=inputs_masked/10000,
    preds=preds/10000,
    n_visible=8,
)

### Inférence depuis le hdf5

In [ ]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 

from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from lib.data_utils import pad_collate

from typing import Any, Dict, Literal
import math
import torch
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib import config_utils
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # ou FutureWarning, UserWarning, ...

In [ ]:
def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch

In [ ]:
item = 500
# t_sampled = [2, 4, 6, 8, 9, 10, 13]
# t_masked = {"indices_masked": [1, 2, 4, 6]}

t_sampled = [0, 1, 2, 3, 4, 5, 6]
t_masked = {"indices_masked": [1, 2, 3, 4, 5]}

phase = "test"
BRIGHTNESS_FACTOR = 3
# Setup configuration
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)

cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl_hdf5 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample_hdf5 = dl_hdf5.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample_hdf5["info"])

In [ ]:
batch_hdf5 = sample_to_batch(sample_hdf5)
batch_hdf5["position_days"] = batch_from_files["position_days"]
batch_hdf5, y_pred_hdf5, att = inference_imputation.impute_sample(
    batch=batch_hdf5,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_hdf5["y"][0],
    inputs=batch_hdf5["x"][0],
    preds=y_pred_hdf5[0],
    n_visible=8,
)

In [ ]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [ ]:
batch_from_files["masks_valid_obs"]

In [ ]:
batch_hdf5["masks_valid_obs"]

In [ ]:
def stats(tensor):
    return f"Min={tensor.min().item():.3f}, Max={tensor.max().item():.3f}, Mean={tensor.float().mean().item():.3f}, Uniques={len(tensor.unique())}"

print("=== STATISTIQUES DES TENSEURS CRITIQUES ===")
for k in ["x", "y", "masks", "cloud_mask", "position_days", "days"]:
    print(f"\n--- {k.upper()} ---")
    print(f"HDF5 : {stats(batch_hdf5[k])}")
    print(f"TIF  : {stats(batch_from_files[k])}")
    if k in ["masks", "cloud_mask", "position_days", "days"]:
        print(f"HDF5 Jours/Vals : {batch_hdf5[k].flatten()[:10].tolist()}")
        print(f"TIF  Jours/Vals : {batch_from_files[k].flatten()[:10].tolist()}")

In [5]:
import json
from pathlib import Path
from dataloader_CIRCA.datasets.cr_metrics import display_metrics

In [7]:
metrics_all_bands_sar_closest_mix_fully_masked = {'ssim': 0.6575257778167725, 'ssim_band_0': 0.47450733184814453, 'ssim_band_1': 0.5389295220375061, 'ssim_band_2': 0.5555816292762756, 'ssim_band_3': 0.6726475358009338, 'ssim_band_4': 0.7396793365478516, 'ssim_band_5': 0.7575511336326599, 'ssim_band_6': 0.6673001646995544, 'ssim_band_7': 0.7601486444473267, 'ssim_band_8': 0.7114553451538086, 'ssim_band_9': 0.6974585056304932, 'ssim_images_occluded_input_pixels': 0.44890615344047546, 'ssim_images_occluded_input_pixels_band_0': 0.33172914385795593, 'ssim_images_occluded_input_pixels_band_1': 0.3691309094429016, 'ssim_images_occluded_input_pixels_band_2': 0.38005203008651733, 'ssim_images_occluded_input_pixels_band_3': 0.462016761302948, 'ssim_images_occluded_input_pixels_band_4': 0.5002996921539307, 'ssim_images_occluded_input_pixels_band_5': 0.5051631927490234, 'ssim_images_occluded_input_pixels_band_6': 0.45085209608078003, 'ssim_images_occluded_input_pixels_band_7': 0.5156282186508179, 'ssim_images_occluded_input_pixels_band_8': 0.49317413568496704, 'ssim_images_occluded_input_pixels_band_9': 0.4810110926628113, 'ssim_images_observed_input_pixels': 0.7373320460319519, 'ssim_images_observed_input_pixels_band_0': 0.5294774174690247, 'ssim_images_observed_input_pixels_band_1': 0.6041911244392395, 'ssim_images_observed_input_pixels_band_2': 0.622512698173523, 'ssim_images_observed_input_pixels_band_3': 0.7534624338150024, 'ssim_images_observed_input_pixels_band_4': 0.8313288688659668, 'ssim_images_observed_input_pixels_band_5': 0.8536627292633057, 'ssim_images_observed_input_pixels_band_6': 0.7497381567955017, 'ssim_images_observed_input_pixels_band_7': 0.8538660407066345, 'ssim_images_observed_input_pixels_band_8': 0.7954583168029785, 'ssim_images_observed_input_pixels_band_9': 0.7796279788017273, 'psnr': 29.44004249572754, 'psnr_occluded_input_pixels': 19.98686981201172, 'psnr_observed_input_pixels': 38.20268249511719, 'mae': 570.4148559570312, 'mae_occluded_input_pixels': 1471.2137451171875, 'mae_observed_input_pixels': 87.29129791259766, 'r2': 0.8031491637229919, 'r2_occluded_input_pixels': 0.5997797250747681, 'r2_observed_input_pixels': 0.9817852973937988, 'mse': 3467534.75, 'mse_occluded_input_pixels': 8371410.5, 'mse_observed_input_pixels': 16456.080078125, 'rmse': 853.0615234375, 'rmse_occluded_input_pixels': 1838.6251220703125, 'rmse_observed_input_pixels': 125.2860107421875, 'psnr_band_0': 31.54471206665039, 'psnr_band_0_occluded_input_pixels': 23.94414520263672, 'psnr_band_0_observed_input_pixels': 38.705543518066406, 'mae_band_0': 504.9038391113281, 'mae_band_0_occluded_input_pixels': 1185.3460693359375, 'mae_band_0_observed_input_pixels': 87.123046875, 'r2_band_0': 0.609946072101593, 'r2_band_0_occluded_input_pixels': 0.32840031385421753, 'r2_band_0_observed_input_pixels': 0.8650376796722412, 'mse_band_0': 3229828.0, 'mse_band_0_occluded_input_pixels': 7359927.5, 'mse_band_0_observed_input_pixels': 18188.90234375, 'rmse_band_0': 785.3751831054688, 'rmse_band_0_occluded_input_pixels': 1564.970703125, 'rmse_band_0_observed_input_pixels': 123.4986572265625, 'psnr_band_1': 32.382957458496094, 'psnr_band_1_occluded_input_pixels': 23.511524200439453, 'psnr_band_1_observed_input_pixels': 40.71519470214844, 'mae_band_1': 492.55078125, 'mae_band_1_occluded_input_pixels': 1221.010986328125, 'mae_band_1_observed_input_pixels': 65.7852783203125, 'r2_band_1': 0.669278621673584, 'r2_band_1_occluded_input_pixels': 0.36549845337867737, 'r2_band_1_observed_input_pixels': 0.9303151965141296, 'mse_band_1': 3213479.75, 'mse_band_1_occluded_input_pixels': 7408137.0, 'mse_band_1_observed_input_pixels': 10387.78125, 'rmse_band_1': 767.1087036132812, 'rmse_band_1_occluded_input_pixels': 1581.97998046875, 'rmse_band_1_observed_input_pixels': 96.50984954833984, 'psnr_band_2': 30.591909408569336, 'psnr_band_2_occluded_input_pixels': 21.887449264526367, 'psnr_band_2_observed_input_pixels': 38.46371841430664, 'mae_band_2': 530.581298828125, 'mae_band_2_occluded_input_pixels': 1294.506103515625, 'mae_band_2_observed_input_pixels': 87.56938171386719, 'r2_band_2': 0.6939961314201355, 'r2_band_2_occluded_input_pixels': 0.39292776584625244, 'r2_band_2_observed_input_pixels': 0.9387997984886169, 'mse_band_2': 3314155.75, 'mse_band_2_occluded_input_pixels': 7669785.0, 'mse_band_2_observed_input_pixels': 18643.701171875, 'rmse_band_2': 813.50830078125, 'rmse_band_2_occluded_input_pixels': 1667.960693359375, 'rmse_band_2_observed_input_pixels': 126.3260726928711, 'psnr_band_3': 31.193485260009766, 'psnr_band_3_occluded_input_pixels': 21.959110260009766, 'psnr_band_3_observed_input_pixels': 39.723907470703125, 'mae_band_3': 535.5280151367188, 'mae_band_3_occluded_input_pixels': 1351.5706787109375, 'mae_band_3_observed_input_pixels': 78.7990951538086, 'r2_band_3': 0.6994869709014893, 'r2_band_3_occluded_input_pixels': 0.38408559560775757, 'r2_band_3_observed_input_pixels': 0.9527727365493774, 'mse_band_3': 3351473.25, 'mse_band_3_occluded_input_pixels': 7887098.0, 'mse_band_3_observed_input_pixels': 13457.1865234375, 'rmse_band_3': 803.96875, 'rmse_band_3_occluded_input_pixels': 1693.9957275390625, 'rmse_band_3_observed_input_pixels': 107.94358825683594, 'psnr_band_4': 29.375194549560547, 'psnr_band_4_occluded_input_pixels': 19.695980072021484, 'psnr_band_4_observed_input_pixels': 38.42339324951172, 'mae_band_4': 605.2877197265625, 'mae_band_4_occluded_input_pixels': 1594.8564453125, 'mae_band_4_observed_input_pixels': 89.70805358886719, 'r2_band_4': 0.7261341214179993, 'r2_band_4_occluded_input_pixels': 0.38197311758995056, 'r2_band_4_observed_input_pixels': 0.970058262348175, 'mse_band_4': 3680270.75, 'mse_band_4_occluded_input_pixels': 8999070.0, 'mse_band_4_observed_input_pixels': 16058.1494140625, 'rmse_band_4': 875.6072998046875, 'rmse_band_4_occluded_input_pixels': 1909.6031494140625, 'rmse_band_4_observed_input_pixels': 123.20208740234375, 'psnr_band_5': 29.177583694458008, 'psnr_band_5_occluded_input_pixels': 18.532934188842773, 'psnr_band_5_observed_input_pixels': 39.9881706237793, 'mae_band_5': 633.339111328125, 'mae_band_5_occluded_input_pixels': 1738.545654296875, 'mae_band_5_observed_input_pixels': 74.02066802978516, 'r2_band_5': 0.7470444440841675, 'r2_band_5_occluded_input_pixels': 0.4013735353946686, 'r2_band_5_observed_input_pixels': 0.9845567941665649, 'mse_band_5': 3966721.25, 'mse_band_5_occluded_input_pixels': 9857612.0, 'mse_band_5_observed_input_pixels': 10613.603515625, 'rmse_band_5': 919.9168701171875, 'rmse_band_5_occluded_input_pixels': 2059.0537109375, 'rmse_band_5_observed_input_pixels': 101.47047424316406, 'psnr_band_6': 27.100282669067383, 'psnr_band_6_occluded_input_pixels': 18.227092742919922, 'psnr_band_6_observed_input_pixels': 35.02191162109375, 'mae_band_6': 681.5897827148438, 'mae_band_6_occluded_input_pixels': 1775.6956787109375, 'mae_band_6_observed_input_pixels': 129.9566650390625, 'r2_band_6': 0.7345524430274963, 'r2_band_6_occluded_input_pixels': 0.40230393409729004, 'r2_band_6_observed_input_pixels': 0.9613139033317566, 'mse_band_6': 4006180.25, 'mse_band_6_occluded_input_pixels': 10069322.0, 'mse_band_6_observed_input_pixels': 33446.59375, 'rmse_band_6': 963.818603515625, 'rmse_band_6_occluded_input_pixels': 2092.998779296875, 'rmse_band_6_observed_input_pixels': 180.1871795654297, 'psnr_band_7': 28.723520278930664, 'psnr_band_7_occluded_input_pixels': 18.414737701416016, 'psnr_band_7_observed_input_pixels': 38.82844543457031, 'mae_band_7': 650.6193237304688, 'mae_band_7_occluded_input_pixels': 1774.33203125, 'mae_band_7_observed_input_pixels': 86.89479064941406, 'r2_band_7': 0.7513291239738464, 'r2_band_7_occluded_input_pixels': 0.4163385331630707, 'r2_band_7_observed_input_pixels': 0.9825872778892517, 'mse_band_7': 4037286.25, 'mse_band_7_occluded_input_pixels': 10141931.0, 'mse_band_7_observed_input_pixels': 13668.466796875, 'rmse_band_7': 933.1566772460938, 'rmse_band_7_occluded_input_pixels': 2088.770751953125, 'rmse_band_7_observed_input_pixels': 115.63949584960938, 'psnr_band_8': 29.808231353759766, 'psnr_band_8_occluded_input_pixels': 20.2741756439209, 'psnr_band_8_observed_input_pixels': 38.8962287902832, 'mae_band_8': 584.607666015625, 'mae_band_8_occluded_input_pixels': 1532.274658203125, 'mae_band_8_observed_input_pixels': 86.3929443359375, 'r2_band_8': 0.7416364550590515, 'r2_band_8_occluded_input_pixels': 0.42873525619506836, 'r2_band_8_observed_input_pixels': 0.9698033332824707, 'mse_band_8': 3464896.75, 'mse_band_8_occluded_input_pixels': 8462217.0, 'mse_band_8_observed_input_pixels': 14370.9609375, 'rmse_band_8': 841.888916015625, 'rmse_band_8_occluded_input_pixels': 1830.302001953125, 'rmse_band_8_observed_input_pixels': 116.39157104492188, 'psnr_band_9': 30.679943084716797, 'psnr_band_9_occluded_input_pixels': 21.732160568237305, 'psnr_band_9_observed_input_pixels': 38.52128219604492, 'mae_band_9': 485.14471435546875, 'mae_band_9_occluded_input_pixels': 1244.0062255859375, 'mae_band_9_observed_input_pixels': 86.66300201416016, 'r2_band_9': 0.7544114589691162, 'r2_band_9_occluded_input_pixels': 0.46637412905693054, 'r2_band_9_observed_input_pixels': 0.9631736874580383, 'mse_band_9': 2411051.0, 'mse_band_9_occluded_input_pixels': 5858994.5, 'mse_band_9_observed_input_pixels': 15725.40234375, 'rmse_band_9': 710.3359985351562, 'rmse_band_9_occluded_input_pixels': 1519.05224609375, 'rmse_band_9_observed_input_pixels': 121.71530151367188, 'sam': 0.04546818882226944, 'sam_occluded_input_pixels': 0.09367521107196808, 'sam_observed_input_pixels': 0.035827673971652985}
display_metrics(metrics_all_bands_sar_closest_mix_fully_masked)

                   Cloud Reconstruction Metrics                   
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃      Metric ┃   Overall    ┃ Occluded Pixels ┃ Observed Pixels ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│         MAE │   570.4149   │    1471.2137    │     87.2913     │
│  MAE_BAND_0 │   504.9038   │    1185.3461    │     87.1230     │
│  MAE_BAND_1 │   492.5508   │    1221.0110    │     65.7853     │
│  MAE_BAND_2 │   530.5813   │    1294.5061    │     87.5694     │
│  MAE_BAND_3 │   535.5280   │    1351.5707    │     78.7991     │
│  MAE_BAND_4 │   605.2877   │    1594.8564    │     89.7081     │
│  MAE_BAND_5 │   633.3391   │    1738.5457    │     74.0207     │
│  MAE_BAND_6 │   681.5898   │    1775.6957    │    129.9567     │
│  MAE_BAND_7 │   650.6193   │    1774.3320    │     86.8948     │
│  MAE_BAND_8 │   584.6077   │    1532.2747    │     86.3929     │
│  MAE_BAND_9 │   485.1447   │    1244.0062    │     86.6630     │
│         MSE │ 3467534.7500 │  8371410.5000   │   16456.0801    │
│  MSE_BAND_0 │ 3229828.0000 │  7359927.5000   │   18188.9023    │
│  MSE_BAND_1 │ 3213479.7500 │  7408137.0000   │   10387.7812    │
│  MSE_BAND_2 │ 3314155.7500 │  7669785.0000   │   18643.7012    │
│  MSE_BAND_3 │ 3351473.2500 │  7887098.0000   │   13457.1865    │
│  MSE_BAND_4 │ 3680270.7500 │  8999070.0000   │   16058.1494    │
│  MSE_BAND_5 │ 3966721.2500 │  9857612.0000   │   10613.6035    │
│  MSE_BAND_6 │ 4006180.2500 │  10069322.0000  │   33446.5938    │
│  MSE_BAND_7 │ 4037286.2500 │  10141931.0000  │   13668.4668    │
│  MSE_BAND_8 │ 3464896.7500 │  8462217.0000   │   14370.9609    │
│  MSE_BAND_9 │ 2411051.0000 │  5858994.5000   │   15725.4023    │
│        PSNR │   29.4400    │     19.9869     │     38.2027     │
│ PSNR_BAND_0 │   31.5447    │     23.9441     │     38.7055     │
│ PSNR_BAND_1 │   32.3830    │     23.5115     │     40.7152     │
│ PSNR_BAND_2 │   30.5919    │     21.8874     │     38.4637     │
│ PSNR_BAND_3 │   31.1935    │     21.9591     │     39.7239     │
│ PSNR_BAND_4 │   29.3752    │     19.6960     │     38.4234     │
│ PSNR_BAND_5 │   29.1776    │     18.5329     │     39.9882     │
│ PSNR_BAND_6 │   27.1003    │     18.2271     │     35.0219     │
│ PSNR_BAND_7 │   28.7235    │     18.4147     │     38.8284     │
│ PSNR_BAND_8 │   29.8082    │     20.2742     │     38.8962     │
│ PSNR_BAND_9 │   30.6799    │     21.7322     │     38.5213     │
│          R2 │    0.8031    │     0.5998      │     0.9818      │
│   R2_BAND_0 │    0.6099    │     0.3284      │     0.8650      │
│   R2_BAND_1 │    0.6693    │     0.3655      │     0.9303      │
│   R2_BAND_2 │    0.6940    │     0.3929      │     0.9388      │
│   R2_BAND_3 │    0.6995    │     0.3841      │     0.9528      │
│   R2_BAND_4 │    0.7261    │     0.3820      │     0.9701      │
│   R2_BAND_5 │    0.7470    │     0.4014      │     0.9846      │
│   R2_BAND_6 │    0.7346    │     0.4023      │     0.9613      │
│   R2_BAND_7 │    0.7513    │     0.4163      │     0.9826      │
│   R2_BAND_8 │    0.7416    │     0.4287      │     0.9698      │
│   R2_BAND_9 │    0.7544    │     0.4664      │     0.9632      │
│        RMSE │   853.0615   │    1838.6251    │    125.2860     │
│ RMSE_BAND_0 │   785.3752   │    1564.9707    │    123.4987     │
│ RMSE_BAND_1 │   767.1087   │    1581.9800    │     96.5098     │
│ RMSE_BAND_2 │   813.5083   │    1667.9607    │    126.3261     │
│ RMSE_BAND_3 │   803.9688   │    1693.9957    │    107.9436     │
│ RMSE_BAND_4 │   875.6073   │    1909.6031    │    123.2021     │
│ RMSE_BAND_5 │   919.9169   │    2059.0537    │    101.4705     │
│ RMSE_BAND_6 │   963.8186   │    2092.9988    │    180.1872     │
│ RMSE_BAND_7 │   933.1567   │    2088.7708    │    115.6395     │
│ RMSE_BAND_8 │   841.8889   │    1830.3020    │    116.3916     │
│ RMSE_BAND_9 │   710.3360   │    1519.0522    │    121.7153     │
│         SAM │    0.0455    │     0.0937      

In [8]:
metrics_multi_stream = {"ssim": 0.2631974220275879, "ssim_band_0": 0.09146042913198471, "ssim_band_1": 0.0931149274110794, "ssim_band_2": 0.0936676561832428, "ssim_band_3": 0.27696382999420166, "ssim_band_4": 0.2659735083580017, "ssim_band_5": 0.5106817483901978, "ssim_band_6": 0.37453824281692505, "ssim_band_7": 0.3265896737575531, "ssim_band_8": 0.28374943137168884, "ssim_band_9": 0.3152405321598053, "ssim_images_occluded_input_pixels": 0.0842423066496849, "ssim_images_occluded_input_pixels_band_0": 0.1564975380897522, "ssim_images_occluded_input_pixels_band_1": 0.18627102673053741, "ssim_images_occluded_input_pixels_band_2": 0.033320967108011246, "ssim_images_occluded_input_pixels_band_3": 0.06711608916521072, "ssim_images_occluded_input_pixels_band_4": 0.0671796202659607, "ssim_images_occluded_input_pixels_band_5": 0.08889535069465637, "ssim_images_occluded_input_pixels_band_6": 0.06224426254630089, "ssim_images_occluded_input_pixels_band_7": 0.07760000228881836, "ssim_images_occluded_input_pixels_band_8": 0.03736196458339691, "ssim_images_occluded_input_pixels_band_9": 0.06593655794858932, "ssim_images_observed_input_pixels": 0.31100335717201233, "ssim_images_observed_input_pixels_band_0": 0.04032247141003609, "ssim_images_observed_input_pixels_band_1": 0.035289254039525986, "ssim_images_observed_input_pixels_band_2": 0.11167263984680176, "ssim_images_observed_input_pixels_band_3": 0.3405678868293762, "ssim_images_observed_input_pixels_band_4": 0.3263545036315918, "ssim_images_observed_input_pixels_band_5": 0.6384670734405518, "ssim_images_observed_input_pixels_band_6": 0.46865910291671753, "ssim_images_observed_input_pixels_band_7": 0.40236741304397583, "ssim_images_observed_input_pixels_band_8": 0.3567279279232025, "ssim_images_observed_input_pixels_band_9": 0.38960638642311096, "rmse": 1167.86279296875, "rmse_occluded_input_pixels": 1754.04638671875, "rmse_observed_input_pixels": 942.951416015625, "mse": 1489116.875, "mse_occluded_input_pixels": 3317880.75, "mse_observed_input_pixels": 890187.75, "psnr": 18.939008712768555, "psnr_occluded_input_pixels": 15.401638984680176, "psnr_observed_input_pixels": 20.51503562927246, "mae": 930.1181640625, "mae_occluded_input_pixels": 1433.209716796875, "mae_observed_input_pixels": 772.5189208984375, "r2": 0.5585875511169434, "r2_occluded_input_pixels": 0.29638442397117615, "r2_observed_input_pixels": 0.6919621229171753, "rmse_band_0": 1339.01416015625, "rmse_band_0_occluded_input_pixels": 1277.8157958984375, "rmse_band_0_observed_input_pixels": 1355.581298828125, "mse_band_0": 1869573.125, "mse_band_0_occluded_input_pixels": 2090129.75, "mse_band_0_observed_input_pixels": 1844457.375, "psnr_band_0": 17.681575775146484, "psnr_band_0_occluded_input_pixels": 20.90699577331543, "psnr_band_0_observed_input_pixels": 17.372589111328125, "mae_band_0": 1275.2930908203125, "mae_band_0_occluded_input_pixels": 1184.301513671875, "mae_band_0_observed_input_pixels": 1335.3443603515625, "r2_band_0": 0.36439937353134155, "r2_band_0_occluded_input_pixels": 0.0019645323045551777, "r2_band_0_observed_input_pixels": 0.4759373664855957, "rmse_band_1": 1539.9815673828125, "rmse_band_1_occluded_input_pixels": 1486.4327392578125, "rmse_band_1_observed_input_pixels": 1555.990478515625, "mse_band_1": 2461794.0, "mse_band_1_occluded_input_pixels": 2785354.0, "mse_band_1_observed_input_pixels": 2430749.5, "psnr_band_1": 16.458024978637695, "psnr_band_1_occluded_input_pixels": 20.52410125732422, "psnr_band_1_observed_input_pixels": 16.175922393798828, "mae_band_1": 1462.982666015625, "mae_band_1_occluded_input_pixels": 1388.0040283203125, "mae_band_1_observed_input_pixels": 1524.80712890625, "r2_band_1": 0.3120937645435333, "r2_band_1_occluded_input_pixels": 0.001308827893808484, "r2_band_1_observed_input_pixels": 0.40439799427986145, "rmse_band_2": 994.8248291015625, "rmse_band_2_occluded_input_pixels": 1270.715087890625, "rmse_band_2_observed_input_pixels": 863.505615234375, "mse_band_2": 1110109.75, "mse_band_2_occluded_input_pixels": 1996176.0, "mse_band_2_observed_input_pixels": 747612.5, "psnr_band_2": 20.38906478881836, "psnr_band_2_occluded_input_pixels": 18.73948860168457, "psnr_band_2_observed_input_pixels": 21.286014556884766, "mae_band_2": 824.9602661132812, "mae_band_2_occluded_input_pixels": 1106.2777099609375, "mae_band_2_observed_input_pixels": 708.8101806640625, "r2_band_2": 0.30926159024238586, "r2_band_2_occluded_input_pixels": 0.0004478652845136821, "r2_band_2_observed_input_pixels": 0.4454585611820221, "rmse_band_3": 586.4703369140625, "rmse_band_3_occluded_input_pixels": 1128.1033935546875, "rmse_band_3_observed_input_pixels": 356.93377685546875, "mse_band_3": 441335.59375, "mse_band_3_occluded_input_pixels": 1454141.25, "mse_band_3_observed_input_pixels": 128281.6328125, "psnr_band_3": 25.490203857421875, "psnr_band_3_occluded_input_pixels": 19.44233512878418, "psnr_band_3_observed_input_pixels": 28.98221778869629, "mae_band_3": 460.53515625, "mae_band_3_occluded_input_pixels": 990.635986328125, "mae_band_3_observed_input_pixels": 298.2038269042969, "r2_band_3": 0.5746492147445679, "r2_band_3_occluded_input_pixels": 0.02890007197856903, "r2_band_3_observed_input_pixels": 0.8248257040977478, "rmse_band_4": 980.2589721679688, "rmse_band_4_occluded_input_pixels": 1545.2779541015625, "rmse_band_4_observed_input_pixels": 749.9490966796875, "mse_band_4": 1105174.0, "mse_band_4_occluded_input_pixels": 2739197.75, "mse_band_4_observed_input_pixels": 568433.625, "psnr_band_4": 20.633522033691406, "psnr_band_4_occluded_input_pixels": 16.7764892578125, "psnr_band_4_observed_input_pixels": 22.54924201965332, "mae_band_4": 843.5771484375, "mae_band_4_occluded_input_pixels": 1372.6102294921875, "mae_band_4_observed_input_pixels": 665.9219360351562, "r2_band_4": 0.6123539209365845, "r2_band_4_occluded_input_pixels": 0.15123671293258667, "r2_band_4_observed_input_pixels": 0.8239293098449707, "rmse_band_5": 1045.3385009765625, "rmse_band_5_occluded_input_pixels": 2199.750244140625, "rmse_band_5_observed_input_pixels": 498.49578857421875, "mse_band_5": 2070417.25, "mse_band_5_occluded_input_pixels": 6905527.0, "mse_band_5_observed_input_pixels": 264886.5, "psnr_band_5": 21.69490623474121, "psnr_band_5_occluded_input_pixels": 14.630419731140137, "psnr_band_5_observed_input_pixels": 26.240808486938477, "mae_band_5": 844.6580810546875, "mae_band_5_occluded_input_pixels": 2028.0882568359375, "mae_band_5_observed_input_pixels": 432.2288513183594, "r2_band_5": 0.6506040692329407, "r2_band_5_occluded_input_pixels": 0.20475661754608154, "r2_band_5_observed_input_pixels": 0.9223546385765076, "rmse_band_6": 1450.8349609375, "rmse_band_6_occluded_input_pixels": 1815.8790283203125, "rmse_band_6_observed_input_pixels": 1298.34716796875, "mse_band_6": 2198888.75, "mse_band_6_occluded_input_pixels": 3699204.25, "mse_band_6_observed_input_pixels": 1689464.75, "psnr_band_6": 16.920133590698242, "psnr_band_6_occluded_input_pixels": 15.364320755004883, "psnr_band_6_observed_input_pixels": 17.742002487182617, "mae_band_6": 1365.3883056640625, "mae_band_6_occluded_input_pixels": 1625.4881591796875, "mae_band_6_observed_input_pixels": 1258.5811767578125, "r2_band_6": 0.6592318415641785, "r2_band_6_occluded_input_pixels": 0.1921653300523758, "r2_band_6_observed_input_pixels": 0.8799258470535278, "rmse_band_7": 815.5139770507812, "rmse_band_7_occluded_input_pixels": 1693.97607421875, "rmse_band_7_observed_input_pixels": 385.1010437011719, "mse_band_7": 1381003.0, "mse_band_7_occluded_input_pixels": 4632036.5, "mse_band_7_observed_input_pixels": 148570.296875, "psnr_band_7": 24.054643630981445, "psnr_band_7_occluded_input_pixels": 17.525104522705078, "psnr_band_7_observed_input_pixels": 28.29623794555664, "mae_band_7": 645.3292846679688, "mae_band_7_occluded_input_pixels": 1491.532470703125, "mae_band_7_observed_input_pixels": 334.7089538574219, "r2_band_7": 0.6568459272384644, "r2_band_7_occluded_input_pixels": 0.18133634328842163, "r2_band_7_observed_input_pixels": 0.8746964931488037, "rmse_band_8": 1300.9886474609375, "rmse_band_8_occluded_input_pixels": 2131.77587890625, "rmse_band_8_observed_input_pixels": 1001.7229614257812, "mse_band_8": 1810812.75, "mse_band_8_occluded_input_pixels": 5062740.0, "mse_band_8_observed_input_pixels": 1008157.625, "psnr_band_8": 18.007761001586914, "psnr_band_8_occluded_input_pixels": 14.141980171203613, "psnr_band_8_observed_input_pixels": 20.006546020507812, "mae_band_8": 1162.4371337890625, "mae_band_8_occluded_input_pixels": 2001.285400390625, "mae_band_8_observed_input_pixels": 959.9014892578125, "r2_band_8": 0.556447446346283, "r2_band_8_occluded_input_pixels": 0.027365531772375107, "r2_band_8_observed_input_pixels": 0.8577106595039368, "rmse_band_9": 587.3428955078125, "rmse_band_9_occluded_input_pixels": 1304.247314453125, "rmse_band_9_observed_input_pixels": 264.97113037109375, "mse_band_9": 442053.15625, "mse_band_9_occluded_input_pixels": 1814333.875, "mse_band_9_observed_input_pixels": 71265.84375, "psnr_band_9": 25.894081115722656, "psnr_band_9_occluded_input_pixels": 18.03335189819336, "psnr_band_9_observed_input_pixels": 31.601707458496094, "mae_band_9": 416.032470703125, "mae_band_9_occluded_input_pixels": 1143.8792724609375, "mae_band_9_observed_input_pixels": 206.6814727783203, "r2_band_9": 0.613256573677063, "r2_band_9_occluded_input_pixels": 0.026966726407408714, "r2_band_9_observed_input_pixels": 0.8834648728370667, "sam": 0.3639760911464691, "sam_occluded_input_pixels": 0.5155124664306641, "sam_observed_input_pixels": 0.3347795307636261}
display_metrics(metrics_multi_stream)

                   Cloud Reconstruction Metrics                   
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃      Metric ┃   Overall    ┃ Occluded Pixels ┃ Observed Pixels ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│         MAE │   930.1182   │    1433.2097    │    772.5189     │
│  MAE_BAND_0 │  1275.2931   │    1184.3015    │    1335.3444    │
│  MAE_BAND_1 │  1462.9827   │    1388.0040    │    1524.8071    │
│  MAE_BAND_2 │   824.9603   │    1106.2777    │    708.8102     │
│  MAE_BAND_3 │   460.5352   │    990.6360     │    298.2038     │
│  MAE_BAND_4 │   843.5771   │    1372.6102    │    665.9219     │
│  MAE_BAND_5 │   844.6581   │    2028.0883    │    432.2289     │
│  MAE_BAND_6 │  1365.3883   │    1625.4882    │    1258.5812    │
│  MAE_BAND_7 │   645.3293   │    1491.5325    │    334.7090     │
│  MAE_BAND_8 │  1162.4371   │    2001.2854    │    959.9015     │
│  MAE_BAND_9 │   416.0325   │    1143.8793    │    206.6815     │
│         MSE │ 1489116.8750 │  3317880.7500   │   890187.7500   │
│  MSE_BAND_0 │ 1869573.1250 │  2090129.7500   │  1844457.3750   │
│  MSE_BAND_1 │ 2461794.0000 │  2785354.0000   │  2430749.5000   │
│  MSE_BAND_2 │ 1110109.7500 │  1996176.0000   │   747612.5000   │
│  MSE_BAND_3 │ 441335.5938  │  1454141.2500   │   128281.6328   │
│  MSE_BAND_4 │ 1105174.0000 │  2739197.7500   │   568433.6250   │
│  MSE_BAND_5 │ 2070417.2500 │  6905527.0000   │   264886.5000   │
│  MSE_BAND_6 │ 2198888.7500 │  3699204.2500   │  1689464.7500   │
│  MSE_BAND_7 │ 1381003.0000 │  4632036.5000   │   148570.2969   │
│  MSE_BAND_8 │ 1810812.7500 │  5062740.0000   │  1008157.6250   │
│  MSE_BAND_9 │ 442053.1562  │  1814333.8750   │   71265.8438    │
│        PSNR │   18.9390    │     15.4016     │     20.5150     │
│ PSNR_BAND_0 │   17.6816    │     20.9070     │     17.3726     │
│ PSNR_BAND_1 │   16.4580    │     20.5241     │     16.1759     │
│ PSNR_BAND_2 │   20.3891    │     18.7395     │     21.2860     │
│ PSNR_BAND_3 │   25.4902    │     19.4423     │     28.9822     │
│ PSNR_BAND_4 │   20.6335    │     16.7765     │     22.5492     │
│ PSNR_BAND_5 │   21.6949    │     14.6304     │     26.2408     │
│ PSNR_BAND_6 │   16.9201    │     15.3643     │     17.7420     │
│ PSNR_BAND_7 │   24.0546    │     17.5251     │     28.2962     │
│ PSNR_BAND_8 │   18.0078    │     14.1420     │     20.0065     │
│ PSNR_BAND_9 │   25.8941    │     18.0334     │     31.6017     │
│          R2 │    0.5586    │     0.2964      │     0.6920      │
│   R2_BAND_0 │    0.3644    │     0.0020      │     0.4759      │
│   R2_BAND_1 │    0.3121    │     0.0013      │     0.4044      │
│   R2_BAND_2 │    0.3093    │     0.0004      │     0.4455      │
│   R2_BAND_3 │    0.5746    │     0.0289      │     0.8248      │
│   R2_BAND_4 │    0.6124    │     0.1512      │     0.8239      │
│   R2_BAND_5 │    0.6506    │     0.2048      │     0.9224      │
│   R2_BAND_6 │    0.6592    │     0.1922      │     0.8799      │
│   R2_BAND_7 │    0.6568    │     0.1813      │     0.8747      │
│   R2_BAND_8 │    0.5564    │     0.0274      │     0.8577      │
│   R2_BAND_9 │    0.6133    │     0.0270      │     0.8835      │
│        RMSE │  1167.8628   │    1754.0464    │    942.9514     │
│ RMSE_BAND_0 │  1339.0142   │    1277.8158    │    1355.5813    │
│ RMSE_BAND_1 │  1539.9816   │    1486.4327    │    1555.9905    │
│ RMSE_BAND_2 │   994.8248   │    1270.7151    │    863.5056     │
│ RMSE_BAND_3 │   586.4703   │    1128.1034    │    356.9338     │
│ RMSE_BAND_4 │   980.2590   │    1545.2780    │    749.9491     │
│ RMSE_BAND_5 │  1045.3385   │    2199.7502    │    498.4958     │
│ RMSE_BAND_6 │  1450.8350   │    1815.8790    │    1298.3472    │
│ RMSE_BAND_7 │   815.5140   │    1693.9761    │    385.1010     │
│ RMSE_BAND_8 │  1300.9886   │    2131.7759    │    1001.7230    │
│ RMSE_BAND_9 │   587.3429   │    1304.2473    │    264.9711     │
│         SAM │    0.3640    │     0.5155      